<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Energy Project Setup Guide F26 (Fall 2026)
### OPIM 5512 — Applied Data Science · Module 4 · Lab 5 pre-lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/Energy_Project_Setup_F26.ipynb)

***Do NOT click Run all.** Copy each gray cell into **Cloud Shell** (in the Google Cloud Console), top to bottom.*

# 🌐 Your energy project: GCP + GitHub Actions setup
**Dr. Dave Wanik and Rohit Akole** - Dept. Operations and Information Management - University of Connecticut

**Fall 2026 edition**, 10/16/2026. These are **the same steps you already did** for your Craigslist project, in a **new**
Google Cloud project, for a **new** repo: [`drdave-teaching/energy-forecasting`](https://github.com/drdave-teaching/energy-forecasting).
Your Craigslist project keeps running exactly as it is. Keeping energy in its own project means its own bill, its own
permissions, and a clean link for the Lab 6 class leaderboard.

**How to use this guide:** copy each gray code cell and paste it into **Cloud Shell**, top to bottom. (Colab may underline
the gray cells in red: ignore that.) Plan on 30 to 45 minutes. Expect a bug or two and debug them with Claude.

**What you'll have at the end:** four Cloud Functions running every hour in your energy project:

| Minute | Function | What it saves in `gs://YOUR-BUCKET/energy/` |
| --- | --- | --- |
| :05 | `scraper-weather-obs` | the weather that actually happened at Bradley Airport |
| :05 | `scraper-weather-forecast` | the National Weather Service forecast for Bradley, next 24 hours |
| :15 | `scraper-isone` | Connecticut electricity load from ISO New England, plus ISO New England's own forecast |
| :20 | `forecast-24h` | your model's forecast for the next 24 hours (it waits politely until Lab 5 gives it a model) |

## 🤖 Stuck? Ask Claude what to do!

**Want a guide at your side the whole way?** Paste this into Claude before you start:

```
Here is my class's Google Cloud setup guide: https://github.com/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/Energy_Project_Setup_F26.ipynb
Walk me through it one step at a time. Tell me exactly what to paste into Cloud Shell or click
in GitHub, wait for me to paste back what happened, and help me fix any errors before we move on.
```

When something goes red, copy this prompt into Claude, fill in the blanks, and paste in the **whole** error message:

```
I'm setting up an hourly weather and electricity-load pipeline on Google Cloud, deployed with
GitHub Actions, for my OPIM 5512 class. I'm on Step ___ of the energy setup guide (___ step name ___).

I ran this in Cloud Shell (or: this GitHub Actions workflow failed):
___ paste the command or the workflow name ___

This is the full error:
___ paste the error ___

My project ID is ___ and my GitHub repo is ___/___.
What does this error mean, and what exactly should I type or click next?
```

Two rules: do the steps **in order**, and **never paste passwords, keys, or tokens** into any chat (that includes your
ISO Express password: it goes only into GitHub's Secrets box in Step 9c).

# 0. Before You Start: Project, Budget, Repo, ISO Express, Cloud Shell

### 0a. Create your energy project
1. Go to **console.cloud.google.com** with the same personal Gmail as your Craigslist project.
2. Click the project picker at the top, then **New project**, for example `energy-yournetid-f26`. Copy the grey
   **Project ID** under the name box: that ID is what you paste below (Google may add a number, like `energy-dww05002-f26-123456`).
3. Make sure it's linked to your billing account (**Billing > Account management** should list it).

### 0b. Set a budget (1 minute, do not skip)
**Budgets & alerts > Create budget**, scope = this project and all services, amount `300`, keep the 50/90/100% alerts,
**Finish**. These four functions run once an hour and save small CSV files, so the project should cost pennies. A budget
only sends emails; if you ever get one, stop and ask for help.

### 0c. Fork the energy repo
1. Open **github.com/drdave-teaching/energy-forecasting** and click **Fork** (you may rename it, for example `energy-forecasting-f26`).
   `GITHUB_REPO` below must match your fork **exactly**: copy it from your browser's address bar.
2. In your fork, open the **Actions** tab and click **"I understand my workflows, go ahead and enable them."**

Make it a **real fork** ("forked from drdave-teaching/energy-forecasting" under the name), so **Sync fork → Update branch**
can bring you Dr. Dave's updates later.

### 0d. Get your own ISO Express account (free, 5 minutes)
ISO New England runs Connecticut's power grid and publishes its load data through **ISO Express web services**, which need
a free login. Create **your own** account at [iso-ne.com](https://www.iso-ne.com/) (look for **Log in / Register** at the
top, then the ISO Express / web services registration). Never use Dr. Dave's or a classmate's account. Keep your username
and password handy: you'll paste them into GitHub **Secrets** in Step 9c, and nowhere else.

### 0e. Open Cloud Shell
Switch the console's project picker to your **energy** project, then click **`>_`** at the top right. Authorize Cloud Shell
if asked. If Cloud Shell disconnects, paste the Step 1 cell again before anything else.

Example: Dr. Dave's test project is `energy-dww05002-f26` and his fork is `drdww/energy-forecasting`.

# 🚀 1. Variables — Edit These First

Before running anything, open your Cloud Shell (top right in GCP console → terminal icon), then paste and modify this block:

In [ ]:
# ==========================================================
# 🧩 EDITABLE VARIABLES (change the first four lines)
# ==========================================================
PROJECT_ID="YOUR-PROJECT-ID"                # e.g. "energy-dww05002-f26" (your ENERGY project)
REGION="us-central1"                        # keep this
BUCKET_NAME="YOUR-PROJECT-ID"               # easiest: same as your project ID
GITHUB_REPO="YOUR-GITHUB-USER/YOUR-REPO"    # e.g. "drdww/energy-forecasting" (must match your fork exactly)

# Internal naming (leave these alone)
WIF_POOL="github-pool"
WIF_PROVIDER="gh-actions"
RUNTIME_SA_ID="cf-runtime"
DEPLOYER_SA_ID="cf-deployer"
SCHED_SA_ID="scheduler-invoker"
# ==========================================================

# --- Derived values ---
gcloud config set project "${PROJECT_ID}"
PROJECT_NUMBER="$(gcloud projects describe "${PROJECT_ID}" --format='value(projectNumber)')"
RUNTIME_SA="${RUNTIME_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"
DEPLOYER_SA="${DEPLOYER_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"
SCHED_SA="${SCHED_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"
SCHEDULER_AGENT="service-${PROJECT_NUMBER}@gcp-sa-cloudscheduler.iam.gserviceaccount.com"
PRINCIPAL_SET="principalSet://iam.googleapis.com/projects/${PROJECT_NUMBER}/locations/global/workloadIdentityPools/${WIF_POOL}/attribute.repository/${GITHUB_REPO}"

echo "✅ Using project ${PROJECT_ID} (${PROJECT_NUMBER})"
echo "✅ GitHub repo: ${GITHUB_REPO}"

**Check the output.** You should see your project ID followed by a long number in parentheses. If the parentheses are **empty**, stop: either the project ID is wrong, or you have not clicked **Authorize** yet. Fix it and paste the cell again.

Still blank after that? **Ask Claude** (see the prompt at the top) and paste in exactly what Cloud Shell printed.

# ⚙️ 2. Enable Required APIs

(idempotent — safe to re-run)

In [ ]:
gcloud services enable \
  cloudfunctions.googleapis.com \
  run.googleapis.com \
  cloudscheduler.googleapis.com \
  artifactregistry.googleapis.com \
  iam.googleapis.com \
  iamcredentials.googleapis.com \
  cloudbuild.googleapis.com \
  eventarc.googleapis.com \
  storage.googleapis.com \
  serviceusage.googleapis.com


A few more APIs...

In [ ]:
gcloud services enable \
  cloudresourcemanager.googleapis.com \
  pubsub.googleapis.com \
  logging.googleapis.com \
  compute.googleapis.com

# 👥 3. Create Service Accounts

(each Cloud Function will run as the runtime SA, and GitHub deploys using the deployer SA)

In [ ]:
gcloud iam service-accounts create "${RUNTIME_SA_ID}"  --display-name="Cloud Functions Runtime" || true
gcloud iam service-accounts create "${DEPLOYER_SA_ID}" --display-name="GitHub Deployer" || true
gcloud iam service-accounts create "${SCHED_SA_ID}"    --display-name="Scheduler Invoker" || true


# 🔗 4. Workload Identity Federation (WIF) Setup

This connects GitHub Actions → GCP securely (no JSON keys!).

In [ ]:
# Create pool (once per project)
gcloud iam workload-identity-pools create "${WIF_POOL}" \
  --location="global" \
  --display-name="GitHub Actions Pool" || true

# Create provider (connects this repo to GCP)
gcloud iam workload-identity-pools providers create-oidc "${WIF_PROVIDER}" \
  --workload-identity-pool="${WIF_POOL}" \
  --location="global" \
  --display-name="GitHub OIDC Provider" \
  --issuer-uri="https://token.actions.githubusercontent.com" \
  --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository,attribute.ref=assertion.ref" \
  --attribute-condition="attribute.repository=='${GITHUB_REPO}' && attribute.ref=='refs/heads/main'" || true


Grant GitHub Actions access to impersonate your deployer service

In [ ]:
gcloud iam service-accounts add-iam-policy-binding "${DEPLOYER_SA}" \
  --role="roles/iam.workloadIdentityUser" \
  --member="${PRINCIPAL_SET}"


# 🔒 5. Project-Level Roles for the Deployer SA

In [ ]:
for ROLE in roles/cloudfunctions.developer roles/run.admin roles/cloudscheduler.admin roles/artifactregistry.writer roles/serviceusage.serviceUsageAdmin; do
  gcloud projects add-iam-policy-binding "${PROJECT_ID}" \
    --member="serviceAccount:${DEPLOYER_SA}" \
    --role="${ROLE}"
done


# 🧩 6. Service Account-to-Service Account Relationships

Allow deployer to “act as” the runtime SA (needed to deploy Functions):

In [ ]:
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" \
  --member="serviceAccount:${DEPLOYER_SA}" \
  --role="roles/iam.serviceAccountUser"


Allow the Cloud Scheduler agent to mint OIDC tokens for the runtime SA (UPDATED MEMBER TO SCHED_SA):

In [ ]:
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" \
  --member="serviceAccount:${SCHED_SA}" \
  --role="roles/iam.serviceAccountTokenCreator"


### Allow Deployer to act as default service account user

In [ ]:
gcloud iam service-accounts add-iam-policy-binding \
  projects/-/serviceAccounts/${PROJECT_NUMBER}-compute@developer.gserviceaccount.com \
  --member="serviceAccount:${DEPLOYER_SA}" \
  --role="roles/iam.serviceAccountUser"


gcloud iam service-accounts add-iam-policy-binding ${RUNTIME_SA} \
  --member="serviceAccount:${DEPLOYER_SA}" \
  --role="roles/iam.serviceAccountAdmin"

A few more permissions

In [ ]:
# 1) Allow the deployer SA to use/impersonate the runtime SA
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" --member="serviceAccount:${DEPLOYER_SA}" --role="roles/iam.serviceAccountUser" --project="${PROJECT_ID}"

# 2) Allow Cloud Scheduler's *managed* service agent to mint tokens for the runtime SA
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" --member="serviceAccount:${SCHEDULER_AGENT}" --role="roles/iam.serviceAccountTokenCreator" --project="${PROJECT_ID}"

If the second command says the Cloud Scheduler service account **does not exist yet**, that is OK: skip it and keep going. The deploy workflow grants the same permission automatically the first time it runs.

### Added the Create Bucket if not exist condition

In [ ]:
if ! gcloud storage buckets describe "gs://${BUCKET_NAME}" >/dev/null 2>&1; then
  gcloud storage buckets create "gs://${BUCKET_NAME}" --project="${PROJECT_ID}" --location="${REGION}"
fi

#🪣 7. Grant Runtime SA Access to Storage Bucket

In [ ]:
gcloud storage buckets add-iam-policy-binding "gs://${BUCKET_NAME}" \
  --member="serviceAccount:${RUNTIME_SA}" \
  --role="roles/storage.objectAdmin"


# 🧱 8. Verify Everything


In [ ]:
gcloud iam service-accounts list

Verify Workload Identity Federation:

In [ ]:
gcloud iam workload-identity-pools list --location=global
gcloud iam workload-identity-pools providers list --workload-identity-pool="${WIF_POOL}" --location=global

Check IAM relationships:

In [ ]:
gcloud projects get-iam-policy "${PROJECT_ID}" | grep "${DEPLOYER_SA}" -A3
gcloud iam service-accounts get-iam-policy "${RUNTIME_SA}"

# 🧩 9. Add 6 Variables and 2 Secrets to Your GitHub Repo

**9a. Print your values.** Paste this cell into Cloud Shell. It prints the 6 variable names with your values filled in.

In [ ]:
# Uses the variables from Step 1 (Cloud Shell restarted? Paste the Step 1 cell again first.)
WORKLOAD_IDENTITY_PROVIDER="projects/${PROJECT_NUMBER}/locations/global/workloadIdentityPools/${WIF_POOL}/providers/${WIF_PROVIDER}"

if [ -z "${PROJECT_NUMBER}" ]; then
  echo "⚠️  Your project number is blank. Paste the Step 1 cell again, then paste this cell again."
fi

echo "=================================================================="
echo "GitHub > Settings > Secrets and variables > Actions > Variables tab"
echo "Add these 6 (click 'New repository variable' for each one):"
echo "=================================================================="
printf "%-28s %s\n" "BUCKET_NAME"                "${BUCKET_NAME}"
printf "%-28s %s\n" "DEPLOYER_SA"                "${DEPLOYER_SA}"
printf "%-28s %s\n" "PROJECT_ID"                 "${PROJECT_ID}"
printf "%-28s %s\n" "REGION"                     "${REGION}"
printf "%-28s %s\n" "RUNTIME_SA"                 "${RUNTIME_SA}"
printf "%-28s %s\n" "WORKLOAD_IDENTITY_PROVIDER" "${WORKLOAD_IDENTITY_PROVIDER}"
echo
echo "✅ Step 9a done. Next: add these 6 in GitHub (Step 9b), then your 2 ISO secrets (Step 9c)."

**9b. Add the 6 variables.** In your **energy** fork: **Settings**, scroll the left sidebar to **Security and quality**,
**Secrets and variables → Actions**, open the **Variables** tab and click **New repository variable** once per row:

| Name | Value |
| --- | --- |
| `BUCKET_NAME` | from the Cloud Shell printout |
| `DEPLOYER_SA` | from the Cloud Shell printout |
| `PROJECT_ID` | from the Cloud Shell printout |
| `REGION` | `us-central1` |
| `RUNTIME_SA` | from the Cloud Shell printout |
| `WORKLOAD_IDENTITY_PROVIDER` | from the Cloud Shell printout (the long `projects/…/providers/gh-actions` line) |

No `BASE_SITE` this time: that was only for Craigslist.

**9c. Add your 2 ISO Express secrets.** Same page, but the **Secrets** tab this time (**not** Variables: variables print in
public logs, secrets never do). Click **New repository secret** twice:

| Name | Value |
| --- | --- |
| `ISO_NE_USERNAME` | your ISO Express username (Step 0d) |
| `ISO_NE_PASSWORD` | your ISO Express password |

Type the names exactly (all capitals, underscores). Without them, `scraper-isone` still deploys but reports
`"status": "skipped"`, and your model has no live load to forecast from.

# ✅ 10. Test Your Setup

In your energy fork on GitHub, open the **Actions** tab and run each of these once (**Run workflow → Run workflow**):
1. **Deploy scraper-weather-obs**
2. **Deploy scraper-weather-forecast**
3. **Deploy scraper-isone**
4. **Deploy forecast-24h**

Wait for **four green checks** (about 2 to 4 minutes each). Then open the **Deploy scraper-isone** run, last step
(**Smoke invoke**): it should say `"status": "ok"`. `"skipped"` means a missing or misspelled secret (Step 9c): fix it and
run that workflow again. **Deploy forecast-24h** says `"no_model"` until Lab 5. That's expected.

Then confirm from Cloud Shell:

In [ ]:
gcloud functions list --regions="${REGION}"
gcloud scheduler jobs list --location="${REGION}"


You should see all four scheduled jobs enabled:

In [ ]:
scraper-weather-obs-hourly
scraper-weather-forecast-hourly
scraper-isone-hourly
forecast-24h-hourly

# Summary Design

In [ ]:
GitHub (OIDC) ───▶ cf-deployer (Workload Identity User)
                          │
                          ├─ deploys CF Gen2 + Scheduler
                          │
                          ▼
                    cf-runtime (ServiceAccountUser)
                          │
                          ├─ executes function logic
                          ├─ writes to GCS bucket
                          └─ trusted by Scheduler agent (TokenCreator)


# ⚡ Watch Your Bucket Fill Up

You don't have to wait for the hour. In the Cloud Console, search **Cloud Scheduler**, tick the checkbox next to
`scraper-weather-obs-hourly`, and click **Force run** (Cloud Shell alternative:
`gcloud scheduler jobs run scraper-weather-obs-hourly --location="${REGION}"`). Do the same for
`scraper-weather-forecast-hourly` and `scraper-isone-hourly`.

Then open **Cloud Storage > Buckets >** your bucket > **`energy/datasets/`**:

| File | What's in it |
| --- | --- |
| `bdl_weather_obs_master.csv` | one row per hour of Bradley weather (grows every hour) |
| `bdl_weather_forecast_master.csv` | every forecast snapshot: 25 rows per hour (this hour + the next 24) |
| `ct_load_hourly_master.csv` | official metered Connecticut load (arrives 2 to 3 days late, so it starts a few days back) |
| `ct_load_5min_hourly_master.csv` | live Connecticut load, averaged per hour (about an hour behind) |
| `isone_ct_forecast_master.csv` | ISO New England's own Connecticut forecast, every version it publishes |

The untouched downloads are kept in `energy/raw/`. In Lab 5 you train a model and upload it; from then on
`forecast-24h` writes your forecast for the next 24 hours to `energy/forecasts/latest.csv` every hour.

**Account safety:** GitHub only *deploys* your code. The schedule lives on Google Cloud, and nothing in this setup commits
to your GitHub repo on a timer. Keep it that way: do not add workflows that push to your repo on a schedule.